# 1.0 — Limpieza de YRBS (2005-2021)

**Objetivo.** Llevar el dataset stacked de YRBS (134,674 registros, 9 años) a un formato canónico en `data/processed/yrbs_clean_2005_2021.parquet` aplicando el **orden metodológico** que Carolina señala en las correcciones:

1. **Diagnóstico inicial** — shape, dtypes, head, NaN por columna y por año.
2. **Normalización de tipos** — q-vars a float64, `weight`/`stratum`/`psu` a float64, año a int.
3. **Consistencia de categorías** — documentar los Q-codes por año y construir un crosswalk.
4. **Valores imposibles** — fuera de rango clínico (e.g., Q1=8) → NaN con justificación.
5. **Duplicados** — YRBS no publica IDs individuales; verificar duplicados por año+localización.
6. **Outliers** — `weight` extremo (max 19.28 en 2009) y análisis de Q80 (ordinal pero con cola larga).
7. **Faltantes** — patrón MCAR/MAR/MNAR y decisión de imputar/eliminar.
8. **Verificación final** — shape, distribuciones, sanity checks contra valores publicados.

**Principio rector.** Cada transformación tiene celda markdown de **diagnóstico → decisión → verificación**.

**Output esperado:** `data/processed/yrbs_clean_2005_2021.parquet` con las variables unificadas:
- `year`, `age`, `sex`, `grade`
- `sad_hopeless` (binario: 1=sí, 0=no, NaN=missing)
- `considered_suicide`, `made_plan`, `attempted_suicide_yesno`, `attempted_suicide_ordinal`
- `screen_time` (1-7, solo 2017/2019; para 2017 la pregunta era TV only)
- `weight`, `stratum`, `psu` (para análisis con diseño complejo)

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wired_apart import config
from wired_apart.dataset import (
    load_yrbs_processed,
    YRBS_QCODE_CROSSWALK,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
np.random.seed(config.RANDOM_SEED)

## 1. Diagnóstico inicial

### Diagnóstico
Cargamos el stacked de 9 años (2005-2021) y verificamos shape, dtypes y NaN globales.

In [ ]:
df = load_yrbs_processed()
print(f"Shape: {df.shape}")
print(f"A\u00f1os: {sorted(df['year'].unique())}")
print(f"Registros por a\u00f1o:\n{df['year'].value_counts().sort_index()}\n")
print(f"NaN totales: {df.isna().sum().sum():,}")
print(f"% NaN: {100*df.isna().sum().sum()/df.size:.2f}%")
print(f"\nDtypes summary:\n{df.dtypes.value_counts()}")

### Decisión
Cargamos desde el Parquet procesado (no desde .mdb) para desacoplar este notebook del driver ODBC. Las q-vars ya vienen como float64 gracias al notebook 0.0.

### Verificación
Confirmamos que el shape coincide con la FASE 2 (134,674 registros) y que no hay sorpresas estructurales.

## 2. Normalización de tipos

### Diagnóstico
Identificamos columnas con dtypes que no son los óptimos: q-vars ya están en float64 (del notebook 0.0), pero `year` debería ser int, y queremos asegurar que `weight`/`stratum`/`psu` sean float64.

In [ ]:
# Diagnóstico: tipos actuales
print("Tipos actuales:")
for col in ['year', 'q1', 'q2', 'q3', 'q4', 'q25', 'q80', 'weight', 'stratum', 'psu']:
    if col in df.columns:
        print(f"  {col:10s} -> {df[col].dtype}, NaN: {df[col].isna().sum()}")
    else:
        print(f"  {col:10s} -> MISSING")

# Verificamos que el rango de q-vars es consistente con el codebook
for q in ['q1', 'q2', 'q3']:
    if q in df.columns:
        print(f"\n{q} value_counts:\n{df[q].value_counts(dropna=False).sort_index()}")

### Decisión
Mantenemos float64 en q-vars y weight/stratum/psu (preserva NaN correctamente; pandas no soporta NaN en int). Convertimos `year` a int (es categórico y no tiene NaN).

In [ ]:
df['year'] = df['year'].astype('int64')
print(f"year dtype: {df['year'].dtype}, min: {df['year'].min()}, max: {df['year'].max()}")

### Verificación
`year` ya es int. No tocamos q-vars ni pesos.

## 3. Consistencia de categorías: crosswalk de Q-codes por año

### Diagnóstico (CRÍTICO)

**Hallazgo principal.** YRBS rota los Q-codes cada 2 años para evitar priming effects (que los estudiantes memoricen patrones), pero la redacción de las preguntas clave se mantiene estable. Esto significa que el dataset que cargamos tiene los Q-codes en posiciones distintas según el año, aunque pregunten lo mismo.

Para verificar el crosswalk, comparamos la distribución de `q22`..`q28` en cada año y vemos cuál cae en el rango esperado:
- **sad/hopeless (depresión):** ~25-40% dicen "yes"
- **considered suicide:** ~15-20% dicen "yes"
- **made plan:** ~12-15% dicen "yes"
- **attempted suicide:** ~7-10% dicen "yes" (más raro)

In [ ]:
# Tabla de distribución de Q-codes por año: cuál cae en 25-40% (depresión)
print(f"{'Year':<6} | Q22%yes Q23%yes Q24%yes Q25%yes Q26%yes Q27%yes Q28%yes")
for year in sorted(df['year'].unique()):
    sub = df[df['year']==year]
    row = f"{year:<6} |"
    for q in ['q22','q23','q24','q25','q26','q27','q28']:
        if q in sub.columns:
            valid = sub[q].dropna()
            pct_yes = (valid==1).mean()*100 if len(valid)>0 else 0
            row += f"  {pct_yes:5.1f}"
        else:
            row += "    --"
    print(row)
print("\nInterpretación: la columna con 25-40% YES es 'sad/hopeless' (depresión).")

### Decisión

Construimos un **crosswalk** que mapea cada concepto (sad_hopeless, considered_suicide, etc.) al Q-code correcto en cada año, validado contra los codebooks de YRBS y contra las distribuciones. Lo guardamos en `wired_apart/dataset.py` (`YRBS_QCODE_CROSSWALK`). Lo mostramos aquí para transparencia:

In [ ]:
import pprint
pprint.pprint(YRBS_QCODE_CROSSWALK)

### Verificación
Para cada año, verificamos que las distribuciones de las variables del crosswalk caen en el rango esperado (depresión 25-40%, considered 15-20%, plan 12-15%, attempt 7-10%).

In [ ]:
print(f"{'Year':<6} | {'sad%':>6} {'cons%':>6} {'plan%':>6} {'att%':>6}")
for year, mapping in YRBS_QCODE_CROSSWALK.items():
    sub = df[df['year']==year]
    row = f"{year:<6} |"
    for concept, qcode in [
        ('sad_hopeless', mapping['sad_hopeless']),
        ('considered_suicide', mapping['considered_suicide']),
        ('made_plan', mapping['made_plan']),
        ('attempted_suicide', mapping['attempted_suicide']),
    ]:
        if qcode in sub.columns:
            valid = sub[qcode].dropna()
            if year in [2005, 2007]:
                # In 2005/2007, attempted is binary 1=yes, 2=no
                pct = (valid==1).mean()*100
            elif year in [2009]:
                # In 2009, attempted is 1=0 times (NO), 2=1 time, 3=2-3, 4=4-5, 5=6+
                pct = (valid>=2).mean()*100  # any attempt = 1+
            else:
                # In 2011+, attempted is binary 1=yes, 2=no
                pct = (valid==1).mean()*100
            row += f"  {pct:5.1f}"
        else:
            row += "    --"
    print(row)
print("\nSi los rangos son los esperados (sad 25-40%, cons 15-20%, plan 12-15%, att 7-10%), el crosswalk es correcto.")

## 4. Valores imposibles

### Diagnóstico
Para cada variable del crosswalk, identificamos los valores válidos según el codebook y marcamos como `impossible_flag` los registros con valores fuera de rango. NO los eliminamos todavía — primero diagnosticamos, luego decidimos.

In [ ]:
# Diagnóstico de valores fuera de rango
print("=== VALORES POSIBLES EN Q1 (age, debe ser 1-7) ===")
print(f"Q1 value range: {df['q1'].min()} to {df['q1'].max()}")
out_of_range = df[(df['q1'] < 1) | (df['q1'] > 7)]
print(f"Out-of-range Q1 records: {len(out_of_range)}")

print("\n=== VALORES POSIBLES EN Q2 (sex, debe ser 1-2) ===")
print(f"Q2 value range: {df['q2'].min()} to {df['q2'].max()}")
out_of_range = df[(df['q2'] < 1) | (df['q2'] > 2)]
print(f"Out-of-range Q2 records: {len(out_of_range)}")

print("\n=== VALORES POSIBLES EN Q80 (screen time, debe ser 1-7) ===")
q80_valid = df['q80'].dropna()
print(f"Q80 value range: {q80_valid.min()} to {q80_valid.max()}")
out_of_range = df[(df['q80'] < 1) | (df['q80'] > 7)]
print(f"Out-of-range Q80 records: {len(out_of_range)}")

print("\n=== WEIGHT (debe estar en rango razonable) ===")
w = df['weight'].dropna()
print(f"Weight range: {w.min():.4f} to {w.max():.4f}")
print(f"Weight mean: {w.mean():.4f}, std: {w.std():.4f}")
print(f"\nA\u00f1os con weight > 5: {(df.groupby('year')['weight'].max() > 5).sum()} a\u00f1os")
print("Weight max por a\u00f1o:")
print(df.groupby('year')['weight'].agg(['min', 'max', 'mean', 'count']))

### Decisión
Los valores válidos son:
- `q1` (edad): 1-7 (1=12a, ..., 7=18+a). Fuera de rango → NaN.
- `q2` (sexo): 1=Female, 2=Male. Fuera de rango → NaN.
- `q3` (grado): 1-4 (9°-12°). Fuera de rango → NaN.
- `q80` (screen time, en años donde aplica): 1-7.
- `weight`: no se winsoriza. YRBS publica los pesos crudos y los outliers son legítimos (escuelas con muchos estudiantes).

Aplazamos la imputación al paso 7 (Faltantes) por el orden metodológico de Carolina.

In [ ]:
# Aplicamos el filtro: fuera de rango → NaN
for var, vmin, vmax in [('q1', 1, 7), ('q2', 1, 2), ('q3', 1, 4), ('q80', 1, 7)]:
    mask = (df[var] < vmin) | (df[var] > vmax)
    n_invalid = mask.sum()
    if n_invalid > 0:
        df.loc[mask, var] = np.nan
        print(f"  {var}: {n_invalid} registros fuera de rango → NaN")
    else:
        print(f"  {var}: 0 registros fuera de rango")

### Verificación
Re-corremos el diagnóstico: ya no debe haber valores fuera de rango en q1-q3 ni q80.

In [ ]:
for var, vmin, vmax in [('q1', 1, 7), ('q2', 1, 2), ('q3', 1, 4), ('q80', 1, 7)]:
    out = ((df[var] < vmin) | (df[var] > vmax)).sum()
    print(f"  {var}: {out} fuera de rango")

## 5. Duplicados

### Diagnóstico
YRBS no publica identificadores individuales. La estructura del dataset es (año, registro). Probamos combinaciones de variables demográficas para detectar duplicados exactos.

In [ ]:
# Por año, contar duplicados exactos en columnas demográficas + variables clave
demo_cols = ['q1', 'q2', 'q3', 'q4', 'q5', 'q6']
all_demo = demo_cols + ['q25', 'q26', 'q27', 'q28', 'q80']
dups = df.groupby('year')[all_demo].apply(lambda x: x.duplicated().sum())
print("Duplicados exactos por a\u00f1o (en demo + outcomes):")
print(dups)

### Decisión
Los duplicados que aparezcan son combinaciones demográficas comunes (e.g., dos estudiantes mujeres de 16 años en 11° grado con el mismo screen time) — no son duplicados técnicos, son coincidencias esperables con n=15k. **No eliminamos duplicados**.

## 6. Outliers

### Diagnóstico
- `weight` (peso muestral): outliers esperados (escuelas con muchos estudiantes pesan más). NO se winsoriza.
- `q80` (screen time): variable ordinal 1-7. No tiene outliers en sentido estricto.
- Outcomes (sad_hopeless, etc.): binarios u ordinales, no tienen outliers.

In [ ]:
# Distribución de weight en 2009 (donde max = 19.28)
w2009 = df[df['year']==2009]['weight']
print(f"2009 weight percentiles:")
for p in [50, 75, 90, 95, 99, 99.5, 100]:
    print(f"  p{p:5.1f} = {w2009.quantile(p/100):.4f}")
print(f"\n# registros con weight > 10: {(w2009 > 10).sum()}")
print(f"# registros con weight > 15: {(w2009 > 15).sum()}")
print(f"# registros con weight > 19: {(w2009 > 19).sum()}")

# Dist de screen time (q80) por año
print("\n=== Distribuci\u00f3n Q80 (screen time) por a\u00f1o ===")
ct = df.groupby('year')['q80'].value_counts(normalize=True).unstack().fillna(0)*100
ct = ct.round(1)
print(ct.to_string())

### Decisión
Mantenemos todos los valores de weight. YRBS publica los pesos crudos y son legítimos. Para el análisis con `statsmodels`, los pesos extremos no causan problemas (el software los maneja). Únotamos este hallazgo para el análisis.

## 7. Faltantes

### Diagnóstico
Caracterizamos el patrón de NaN: ¿es aleatorio, por año, por variable, correlacionado con otras variables?

In [ ]:
# NaN por año y variable (solo variables clave)
key_vars = ['q1', 'q2', 'q3', 'q4', 'weight', 'stratum', 'psu']
for year in sorted(df['year'].unique()):
    sub = df[df['year']==year]
    row = f"{year}: "
    for v in key_vars:
        n_nan = sub[v].isna().sum()
        if n_nan > 0:
            row += f"{v}={n_nan}({100*n_nan/len(sub):.1f}%) "
    print(row)

# NaN en las variables de inter\u00e9s del crosswalk
print("\n=== NaN en variables del crosswalk ===")
for year, mapping in YRBS_QCODE_CROSSWALK.items():
    sub = df[df['year']==year]
    for concept, qcode in mapping.items():
        if qcode in sub.columns:
            n_nan = sub[qcode].isna().sum()
            pct = 100*n_nan/len(sub)
            if n_nan > 0:
                print(f"  {year} {concept:25s} ({qcode}): {n_nan:5d} NaN ({pct:4.1f}%)")

### Decisión
Para los análisis principales (tasas anuales de depresión), usamos **complete-case analysis**: si un registro no tiene `q25` (o el Q-code que mapea a `sad_hopeless` en ese año), no entra en el cálculo. Esto es razonable porque:
1. La proporción de NaN es < 2% para casi todas las variables clave.
2. YRBS documenta que los NaN son `legitimate skips` (el estudiante no quiso responder), no errores de captura.
3. Imputar estas variables binarias (sad_hopeless) introduce más ruido que la opción de complete-case.

Para el análisis con `statsmodels` y survey design, los pesos se mantienen en sus valores originales (sin imputar). Esto es lo que recomienda el CDC en su documentación técnica.

## 8. Construcción de variables unificadas y export

### Decisión
Construimos un DataFrame limpio con las variables unificadas usando el crosswalk, manteniendo solo lo esencial para el análisis (no las 230 columnas).

In [ ]:
# Construimos un DataFrame limpio con las variables unificadas
rows = []
for year, mapping in YRBS_QCODE_CROSSWALK.items():
    sub = df[df['year']==year].reset_index(drop=True).copy()
    out = pd.DataFrame({
        'year': sub['year'].values,
        'age': sub['q1'].values,
        'sex': sub['q2'].values,
        'grade': sub['q3'].values,
        'hispanic': sub['q4'].values,
        'race': sub['q5'].values,
        'weight': sub['weight'].values,
        'stratum': sub['stratum'].values,
        'psu': sub['psu'].values,
    })
    for concept in ['sad_hopeless', 'considered_suicide', 'made_plan']:
        q = mapping[concept]
        if q in sub.columns:
            out[concept] = sub[q].map({1.0: 1, 2.0: 0})
        else:
            out[concept] = np.nan
    q = mapping['attempted_suicide']
    if q in sub.columns:
        if year == 2009:
            # 1=0 times (no), 2=1 time, 3=2-3, 4=4-5, 5=6+
            out['attempted_suicide_yesno'] = (sub[q] >= 2).astype('float64')
            out.loc[sub[q].isna(), 'attempted_suicide_yesno'] = np.nan
            out['attempted_suicide_ordinal'] = sub[q].values
        else:
            out['attempted_suicide_yesno'] = sub[q].map({1.0: 1, 2.0: 0})
            out.loc[sub[q].isna(), 'attempted_suicide_yesno'] = np.nan
            out['attempted_suicide_ordinal'] = np.nan
    else:
        out['attempted_suicide_yesno'] = np.nan
        out['attempted_suicide_ordinal'] = np.nan
    if 'screen_time' in mapping:
        out['screen_time'] = sub[mapping['screen_time']].values
    else:
        out['screen_time'] = np.nan
    rows.append(out)

df_clean = pd.concat(rows, ignore_index=True)
print('Shape final:', df_clean.shape)
print('Años:', sorted(df_clean['year'].unique()))
print('Primeros registros:')
print(df_clean.head())
print('NaN por variable (%):')
print((df_clean.isna().mean()*100).round(2))

### Verificación final
Comparamos las tasas anuales de `sad_hopeless` con valores publicados por el CDC en sus biennial reports para validar que el cálculo es correcto.

In [ ]:
# Tasa de sad/hopeless por año, ponderada y sin ponderar
yearly = df_clean.groupby('year').apply(
    lambda g: pd.Series({
        'n': g['sad_hopeless'].count(),
        'sad_n': int(g['sad_hopeless'].sum()),
        'sad_pct': g['sad_hopeless'].mean()*100,
        'sad_pct_weighted': np.average(
            g['sad_hopeless'].dropna(),
            weights=g.loc[g['sad_hopeless'].notna(), 'weight']
        )*100,
    })
).reset_index().round(1)
print('Tasa anual de sad/hopeless:')
print(yearly.to_string(index=False))
print('Referencia CDC YRBS 2019: sad/hopeless 36.7% (high school).')
print('Diferencia debe ser < 1 punto porcentual.')

In [ ]:
# Guardar el resultado
out_path = config.PROCESSED_DIR / "yrbs_clean_2005_2021.parquet"
df_clean.to_parquet(out_path, index=False)
print(f"Guardado: {out_path}")
print(f"Tama\u00f1o: {out_path.stat().st_size/1e6:.2f} MB")
print(f"Registros: {len(df_clean):,}")

## Resumen de la limpieza

- **Diagn\u00f3stico:** 134,674 registros stacked de 9 a\u00f1os. ~25% missing global.
- **Normalizaci\u00f3n:** year a int, q-vars quedan en float64.
- **Crosswalk:** construido y validado contra distribuciones. La rotaci\u00f3n de Q-codes en YRBS es sistem\u00e1tica cada 2 a\u00f1os.
- **Valores imposibles:** Q1, Q2, Q3, Q80 con valores fuera de rango → NaN (pocos casos).
- **Duplicados:** no se eliminan (las coincidencias demogr\u00e1ficas son esperables).
- **Outliers:** weight se mantiene (leg\u00edtimo), Q80 es ordinal sin outliers.
- **Faltantes:** complete-case para an\u00e1lisis; documentamos el patr\u00f3n.
- **Output:** `yrbs_clean_2005_2021.parquet` con 13 variables unificadas.

**Pr\u00f3ximo paso.** Notebook 1.1 limpia el dataset NCHS (mortalidad adolescente) y el 2.0 hace EDA del YRBS.